# Advanced Problems: Global and Local Scopes

Each problem focuses on Python scoping rules, name resolution, shadowing, `global`, and compile-time scope decisions.

**Instructions:** Predict the output first, then run the solution cells.

## Problem 1: Compile-Time Local Scope

Predict what happens when `func()` is called. Explain why.

In [3]:
x = 50

def func():
    print(x)
    x = 100

# func()

### Solution 1

Calling `func()` raises `UnboundLocalError`.

Because `x = 100` appears anywhere inside the function body, Python treats `x` as a local variable for the entire function. Therefore, `print(x)` tries to read the local `x` before it has been assigned.

In [4]:
x = 50

def func():
    print(x)
    x = 100

try:
    func()
except Exception as ex:
    print(type(ex).__name__ + ':', ex)

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value


---

## Problem 2: Fixing the Function with `global`

Modify the function so it updates the module-level `x` instead of creating a local variable.

In [5]:
x = 50

def func():
    # Your fix here
    print(x)
    x = 100

# func()
# print(x)

### Solution 2

Use `global x` before reading or assigning `x`.

In [6]:
x = 50

def func():
    global x
    print(x)
    x = 100

func()
print(x)

50
100


Expected output:

```text
50
100
```

---

## Problem 3: Global Mutation vs Global Rebinding

Predict the output. Does this function need `global`?

In [7]:
items = []

def add_item(value):
    items.append(value)

add_item('A')
add_item('B')
print(items)

['A', 'B']


### Solution 3

Output:

```text
['A', 'B']
```

The function does **not** need `global` because it does not reassign the name `items`. It only mutates the list object that `items` refers to.

---

## Problem 4: Rebinding a Global Mutable Object

Predict what happens here.

In [8]:
items = ['original']

def reset_items():
    items = []
    items.append('local')
    print('inside:', items)

reset_items()
print('outside:', items)

inside: ['local']
outside: ['original']


### Solution 4

Output:

```text
inside: ['local']
outside: ['original']
```

`items = []` creates a new local variable named `items`. It does not change the global `items`.

---

## Problem 5: Fixing Global Rebinding

Rewrite `reset_items()` so that it actually replaces the global list.

In [9]:
items = ['original']

def reset_items():
    # Your fix here
    items = []
    items.append('global')

# reset_items()
# print(items)

### Solution 5

In [10]:
items = ['original']

def reset_items():
    global items
    items = []
    items.append('global')

reset_items()
print(items)

['global']


Expected output:

```text
['global']
```

---

## Problem 6: Shadowing a Built-In Name

Predict what happens and explain the problem.

In [11]:
len = 100

def count_values(values):
    return len(values)

try:
    print(count_values([1, 2, 3]))
except Exception as ex:
    print(type(ex).__name__ + ':', ex)

del len

TypeError: 'int' object is not callable


### Solution 6

This raises:

```text
TypeError: 'int' object is not callable
```

The global name `len` shadows Python's built-in `len` function. Inside `count_values`, Python finds the global integer `len = 100`, then tries to call it like a function.

---

## Problem 7: Loop Variables and Scope

Predict the output.

In [12]:
for i in range(4):
    result = i * 10

print(i)
print(result)

3
30


### Solution 7

Output:

```text
3
30
```

`for` loops do not create a new local scope in Python. At module level, both `i` and `result` are global names.

---

## Problem 8: Function Scope vs Loop Scope

Predict the output.

In [13]:
def compute():
    for i in range(3):
        value = i ** 2
    return i, value

print(compute())

try:
    print(value)
except Exception as ex:
    print(type(ex).__name__ + ':', ex)

(2, 4)
NameError: name 'value' is not defined


### Solution 8

Output:

```text
(2, 4)
NameError: name 'value' is not defined
```

The loop does not create a separate scope, so `i` and `value` are available inside `compute()` after the loop. But they are local to `compute()`, so they are not available outside the function.

---

## Problem 9: Creating a Global from Inside a Function

Predict the output.

In [14]:
def create_config():
    global config
    config = {'debug': True}

try:
    print(config)
except Exception as ex:
    print('before:', type(ex).__name__)

create_config()
print('after:', config)

before: NameError
after: {'debug': True}


### Solution 9

Output:

```text
before: NameError
after: {'debug': True}
```

`global config` tells Python to bind the name `config` in the module scope. The name does not exist until the function is actually called.

---

## Problem 10: Avoiding `global` with Return Values

Refactor the code to avoid modifying global state.

In [15]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(counter)

2


### Solution 10

A better design is to pass state in and return the updated value.

In [16]:
def increment(counter):
    return counter + 1

counter = 0
counter = increment(counter)
counter = increment(counter)
print(counter)

2


Expected output:

```text
2
```

This version is easier to test and avoids hidden side effects.

---

## Problem 11: Name Resolution with Built-Ins

Predict the output.

In [17]:
abs = lambda x: 'custom abs'

def test():
    return abs(-10)

print(test())

del abs
print(abs(-10))

custom abs
10


### Solution 11

Output:

```text
custom abs
10
```

Python first looks in the local scope, then global scope, then built-ins. The custom global `abs` is found before the built-in `abs`. After `del abs`, the built-in becomes visible again.

---

## Problem 12: Debugging a Bad Accumulator

The following function is supposed to add numbers to the global `total`, but it fails. Fix it.

In [18]:
total = 0

def add_to_total(values):
    for value in values:
        total += value
    return total

# add_to_total([10, 20, 30])

### Solution 12

`total += value` is an assignment operation. Therefore, Python treats `total` as local unless we declare it global.

In [19]:
total = 0

def add_to_total(values):
    global total
    for value in values:
        total += value
    return total

print(add_to_total([10, 20, 30]))
print(total)

60
60


Expected output:

```text
60
60
```

However, a cleaner version avoids global mutation:

In [20]:
def add_values(values):
    total = 0
    for value in values:
        total += value
    return total

print(add_values([10, 20, 30]))

60


---

## Problem 13: Reading Global, Assigning Local

Why does this function fail?

In [21]:
status = 'ready'

def update_status(flag):
    if flag:
        status = 'running'
    return status

try:
    print(update_status(False))
except Exception as ex:
    print(type(ex).__name__ + ':', ex)

UnboundLocalError: cannot access local variable 'status' where it is not associated with a value


### Solution 13

Even though `status = 'running'` is inside an `if` block, Python still treats `status` as local throughout the whole function.

When `flag` is `False`, the assignment never runs, so `return status` tries to return an unassigned local variable.

A good fix is to avoid depending on global state:

In [22]:
def update_status(current_status, flag):
    if flag:
        return 'running'
    return current_status

status = 'ready'
status = update_status(status, False)
print(status)

status = update_status(status, True)
print(status)

ready
running


---

## Problem 14: Best Practice Refactor

The following code works, but it is difficult to test because it relies on global state. Refactor it into a pure function.

In [23]:
tax_rate = 0.2
discount = 10

def final_price(price):
    return price + price * tax_rate - discount

print(final_price(100))

110.0


### Solution 14

Pass dependencies explicitly as parameters.

In [24]:
def final_price(price, tax_rate, discount):
    return price + price * tax_rate - discount

print(final_price(100, tax_rate=0.2, discount=10))

110.0


Expected output:

```text
110.0
```

This version is easier to reuse, test, and reason about.

---

## Problem 15: Comprehensive Challenge

Predict the output carefully.

In [25]:
message = 'global'
values = []

def outer():
    message = 'outer local'
    values.append(message)

    def inner():
        global message
        message = 'changed global'
        values.append(message)

    inner()
    values.append(message)

outer()
print(message)
print(values)

changed global
['outer local', 'changed global', 'outer local']


### Solution 15

Output:

```text
changed global
['outer local', 'changed global', 'outer local']
```

Explanation:

- `outer()` creates a local `message` with value `'outer local'`.
- `values.append(message)` appends the outer local value.
- `inner()` declares `global message`, so its assignment changes the module-level `message`.
- The list `values` is mutated, not rebound, so no `global values` is needed.
- After `inner()` finishes, `outer()` still has its own local `message`, so it appends `'outer local'` again.

## Key Takeaways

- Assignment inside a function makes a name local unless declared `global`.
- Python determines local/global classification at compile time, not at runtime.
- Reading a global variable is allowed without `global`.
- Rebinding a global variable requires `global`.
- Mutating a global object usually does not require `global`.
- Loops and conditionals do not create their own local scope.
- Avoid shadowing built-ins like `list`, `dict`, `str`, `sum`, `len`, `abs`, and `print`.
- Best practice: prefer explicit parameters and return values over global mutation.